<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h1>Notebook Modelisation - ALS (Spark MLlib)</h1>
<strong>Alternating Least Squares matrix factorization</strong>
<p>Entraine un modele ALS sur les splits temporels preparés dans l'EDA<br>
selectionne les hyperparametres sur le jeu `validation` et evalue sur le jeu`test`.</p>
</div>

In [5]:
from pathlib import Path

import pandas as pd
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import functions as F

import src.utils as utils

In [ ]:
# Sélection de la source : "processed_small" ou "processed_big"
DATA_SOURCE = "processed_small"  
RANDOM_SEED = 42
TOP_N = 10

if DATA_SOURCE not in {"processed_small", "processed_big"}:
    raise ValueError(
        f"DATA_SOURCE invalide pour ce notebook: {DATA_SOURCE}. Utiliser processed_small ou processed_big."
    )

DATA_SIZE = DATA_SOURCE.replace("processed_", "")

# Grille simple pour debuter
GRID_RANK = [20, 40]
GRID_REG_PARAM = [0.05, 0.1]
GRID_MAX_ITER = [10, 15]

In [7]:
spark = utils.create_spark_session()
project_root = utils.get_project_root()

# Meme procédé que l'EDA: on resout les chemins a partir de DATA_SOURCE
dataset_format, path_ratings, path_movies = utils.resolve_data_source_paths(
    DATA_SOURCE, project_root=project_root
)
if dataset_format != "parquet":
    raise ValueError(
        f"Ce notebook ALS attend une source processed_* en parquet, recu: {DATA_SOURCE} ({dataset_format})"
    )

processed_root = project_root / "data" / "processed" / DATA_SIZE
split_root = processed_root / "splits_temporal"

required_paths = [
    path_ratings,
    path_movies,
    split_root / "train",
    split_root / "validation",
    split_root / "test",
]

missing = [p for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Artefacts manquants: {missing}")

df_movies_clean = spark.read.parquet(path_movies.as_posix())
df_train = spark.read.parquet((split_root / "train").as_posix())
df_val = spark.read.parquet((split_root / "validation").as_posix())
df_test = spark.read.parquet((split_root / "test").as_posix())

for name in ["df_train", "df_val", "df_test"]:
    df = globals()[name]
    globals()[name] = df.select(
        F.col("userId").cast("int").alias("userId"),
        F.col("movieId").cast("int").alias("movieId"),
        F.col("rating").cast("float").alias("rating"),
        F.col("timestamp").cast("long").alias("timestamp"),
    )

print(f"Artefacts charges avec succes depuis DATA_SOURCE={DATA_SOURCE}.")

26/04/14 23:31:57 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.command

Artefacts charges avec succes depuis DATA_SOURCE=processed_small.


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Chargement des artefacts</h3>
<i>artefact : fichier produit par une étape du pipeline et réutilisé plus tard.</i><br>
Si le message affiche <b>"Artefacts charges avec succes"</b>, la base experimentale est prete.

Verification implicite:
- Les fichiers nettoyes et les splits temporels existent bien.
- Les colonnes userId, movieId, rating, timestamp ont ete castees au bon type pour ALS.

En cas d'erreur, il faut relancer le notebook EDA pour regenerer les artefacts.
</div>

In [8]:
print("--- Tailles des splits ---")
print(f"Train: {df_train.count()}")
print(f"Validation: {df_val.count()}")
print(f"Test: {df_test.count()}")

print("\n--- Cardinalites train ---")
print(f"Users train: {df_train.select('userId').distinct().count()}")
print(f"Items train: {df_train.select('movieId').distinct().count()}")

--- Tailles des splits ---
Train: 80669
Validation: 10084
Test: 10083

--- Cardinalites train ---
Users train: 522
Items train: 7867


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Taille des splits</h3>
Cette sortie confirme la proportion train/validation/test et la cardinalite users/items en train.

Comment interpreter:
- Train doit etre majoritaire pour apprendre correctement.
- Validation sert au choix des hyperparametres.
- Test reste strictement reserve a l'evaluation finale.

Si les cardinalites sont tres faibles, il faut reduire la complexite du modele (rank, iterations).
</div>

In [ ]:
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

# Initialisations pour la recherche par grille
results = []
best_rmse = float("inf")
best_params = None


# ALS, algorithme de factorisation matricielle
for rank in GRID_RANK:
    for reg_param in GRID_REG_PARAM:
        for max_iter in GRID_MAX_ITER:
            als = ALS(
                userCol="userId",
                itemCol="movieId",
                ratingCol="rating",
                rank=rank,
                regParam=reg_param,
                maxIter=max_iter,
                coldStartStrategy="drop",
                nonnegative=True,
                seed=RANDOM_SEED,
            )

            model = als.fit(df_train)
            pred_val = model.transform(df_val).dropna(subset=["prediction"])
            rmse_val = evaluator.evaluate(pred_val)

            # Enregistrement des resultats
            results.append({
                "rank": rank,
                "regParam": reg_param,
                "maxIter": max_iter,
                "rmse_val": rmse_val,
            })

            # Mise a jour du score et des params, si meilleur que les precedents
            if rmse_val < best_rmse:
                best_rmse = rmse_val
                best_params = (rank, reg_param, max_iter)

# Affichage des resultats de la grille et du meilleur score
df_grid = pd.DataFrame(results).sort_values("rmse_val")
display(df_grid)
print(f"Meilleurs params (val): rank={best_params[0]}, regParam={best_params[1]}, maxIter={best_params[2]}")
print(f"RMSE validation: {best_rmse:.4f}")

26/04/14 23:32:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


,rank,regParam,maxIter,rmse_val
6,40,0.10,10,0.940857
7,40,0.10,15,0.941185
3,20,0.10,15,0.944285
2,20,0.10,10,0.945507
1,20,0.05,15,0.992561
5,40,0.05,15,0.998106
0,20,0.05,10,1.008244
4,40,0.05,10,1.020663


Meilleurs params (val): rank=40, regParam=0.1, maxIter=10
RMSE validation: 0.9409


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Tuning ALS sur validation</h3>
Le tableau affiche chaque combinaison (rank, regParam, maxIter) et son RMSE validation.

Regle de lecture:
- Plus le RMSE est bas, meilleure est la prediction de note sur validation.
- La ligne retenue devient la configuration du modele final.

Comment choisir en pratique:
- Commencer par le plus petit RMSE.
- En cas de scores tres proches, preferer la configuration la plus simple (rank plus faible, moins d'iterations) pour limiter le risque de surapprentissage et le temps de calcul.

Attention:
- Un bon score validation ne garantit pas toujours un bon score test.
- Le test final sert a verifier la generalisation.
- Si train est bon mais validation/test se degradent, reduire la complexite ou renforcer le filtrage des utilisateurs/items tres rares.
</div>

In [ ]:
'''
Re-entraine sur train + validation avec les meilleurs parmètres
entraîne le modèle final avec un maximum de données non-test
pour améliorer la généralisation.
'''
df_train_val = df_train.unionByName(df_val)

best_rank, best_reg_param, best_max_iter = best_params
als_final = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=best_rank,
    regParam=best_reg_param,
    maxIter=best_max_iter,
    coldStartStrategy="drop",
    nonnegative=True,
    seed=RANDOM_SEED,
)

model_final = als_final.fit(df_train_val)

# Evaluation finale sur le test set
pred_test = model_final.transform(df_test).dropna(subset=["prediction"])
rmse_test = evaluator.evaluate(pred_test)

print(f"RMSE test (modele final): {rmse_test:.4f}")

RMSE test (modele final): 0.9020


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - RMSE test</h3>
Ce score est la mesure principale de performance hors echantillon.

Interpretation:
- Plus le RMSE test est bas, plus la prediction des notes est precise.
- Si RMSE du jeu de test est nettement pire que celle du jeu de validation<br>
  il peut y avoir surapprentissage.

Ordres de grandeur utiles:
- Une baisse de RMSE de 0.01 a 0.03 est deja interessante sur MovieLens.
- Une baisse plus forte indique souvent un vrai gain de calibration du modele.

Limite a garder en tete:
- Le RMSE evalue la prediction de note, pas directement la qualite du top-N.
- C'est pour cela qu'on ajoute ensuite Precision@K, Recall@K et Coverage@K.

Ce score servira de reference pour comparer ensuite contenu et KNN.
</div>

In [11]:
# Recommandations Top-N pour quelques utilisateurs
sample_users = [r.userId for r in df_train.select("userId").distinct().limit(5).collect()]
df_users = spark.createDataFrame([(u,) for u in sample_users], ["userId"])

df_reco = model_final.recommendForUserSubset(df_users, TOP_N)
df_reco_flat = (
    df_reco
    .withColumn("rec", F.explode("recommendations"))
    .select(
        "userId",
        F.col("rec.movieId").alias("movieId"),
        F.col("rec.rating").alias("score"),
    )
    .join(df_movies_clean.select("movieId", "title"), on="movieId", how="left")
    .orderBy("userId", F.desc("score"))
)

df_reco_flat.show(50, truncate=False)

+-------+------+---------+---------------------------------------------------------------------------+
|movieId|userId|score    |title                                                                      |
+-------+------+---------+---------------------------------------------------------------------------+
|59814  |243   |5.3983574|Ex Drummer (2007)                                                          |
|74946  |243   |5.3394585|She's Out of My League (2010)                                              |
|86377  |243   |5.278813 |Louis C.K.: Shameless (2007)                                               |
|138966 |243   |5.2779603|Nasu: Summer in Andalusia (2003)                                           |
|134796 |243   |5.2779603|Bitter Lake (2015)                                                         |
|117531 |243   |5.2779603|Watermark (2014)                                                           |
|86237  |243   |5.2779603|Connections (1978)                             

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Lecture du resultat - Top-N recommandations</h3>
Le tableau liste les films recommandes pour quelques utilisateurs de test.

Interpretation des colonnes:
- userId: utilisateur cible.
- movieId / title: film recommande.
- score: score predit par ALS (plus eleve = recommandation plus forte).

Controle qualite simple:
- Verifier la diversite des titres proposes.
- Verifier que les recommendations semblent plausibles pour chaque utilisateur.
</div>

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<h3>Métriques</h3>
</div>


<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<strong>Precision@K (La pertinence)</strong><br>
Sur les $K$ films recommandés, combien sont réellement pertinents pour l'utilisateur ?<br>
Si on affiche un top 5 sur l'application,<br>
une Precision@5 de 0.8 signifie que 4 films sur 5 plaisent à l'utilisateur.
</div>

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">

<strong>Recall@K (L'exhaustivité)</strong><br>
Sur tous les films que l'utilisateur aime vraiment, quelle proportion ai-je réussi à capturer dans mon top K ?<br>
Contrairement à la précision, le dénominateur est le nombre total de "coups de cœur" possibles de l'utilisateur
</div>

<div style="color: #849dc9; background-color: #223f4d; padding: 10px; border-radius: 8px;">
<strong>Coverage@K (La diversité du catalogue)</strong><br>
La Couverture ne regarde pas l'utilisateur, mais le catalogue.<br>
Sur l'ensemble de la base de films, quel pourcentage est effectivement recommandé au moins une fois dans les tops K des utilisateurs ?<br>
Un modèle peut avoir une excellente Precision@K en recommandant uniquement Titanic et Avatar à tout le monde.<br>
Mais le Coverage sera très faible (proche de 0%).<br>
Permet d'éviter l'effet "bulle" et s'assurer que les films de niche sont aussi proposés
</div>

Precision : $\frac{\text{Nombre de recommandations pertinentes parmi les } K \text{ premières}}{K}$

Recall : $\frac{\text{Nombre de recommandations pertinentes parmi les } K \text{ premières}}{\text{Nombre total d'items pertinents pour cet utilisateur}}$

In [ ]:
# Evaluation Top-K commune (ALS baseline): Precision@K, Recall@K, Coverage@K
# A executer apres l'entrainement de model_final.
K = TOP_N
RELEVANCE_THRESHOLD = 4.0

# 1) Utilisateurs a evaluer: presents a la fois dans train_val et dans test.
# On ne mesure la qualite que la ou une verite terrain existe (interactions test).
df_users_train_val = df_train_val.select("userId").distinct()
df_users_test = df_test.select("userId").distinct()
df_eval_users = df_users_train_val.join(df_users_test, on="userId", how="inner")

# 2) Recommandations ALS Top-K pour ces utilisateurs
df_reco_eval = model_final.recommendForUserSubset(df_eval_users, K)
df_reco_eval_flat = (
    df_reco_eval.withColumn(
        "rec",
        F.explode("recommendations")
    ).select(
        "userId",
        F.col("rec.movieId").cast("int").alias("movieId"),
        F.col("rec.rating").alias("score")
    )
)

# 3) Verite terrain test: item pertinent si rating >= seuil
df_relevant = (
    df_test
    .filter(F.col("rating") >= RELEVANCE_THRESHOLD)
    .select("userId", "movieId")
    .distinct()
)

# 4) True positives = recommandations qui apparaissent aussi dans les items pertinents
df_hits = df_reco_eval_flat.join(df_relevant, on=["userId", "movieId"], how="inner")
df_hits_per_user = df_hits.groupBy("userId").agg(F.count("*").alias("tp"))

# Nombre de recommandations par utilisateur (robuste meme si < K dans certains cas rares)
df_rec_per_user = df_reco_eval_flat.groupBy("userId").agg(F.count("*").alias("n_rec"))

# Nombre d'items pertinents reels par utilisateur
df_rel_per_user = df_relevant.groupBy("userId").agg(F.count("*").alias("n_rel"))

# 5) Precision@K et Recall@K par utilisateur puis moyenne macro
df_user_metrics = (
    df_rec_per_user
    .join(df_rel_per_user, on="userId", how="left")
    .join(df_hits_per_user, on="userId", how="left")
    .fillna({"n_rel": 0, "tp": 0})
    .withColumn("precision_at_k", F.col("tp") / F.col("n_rec"))
    .withColumn(
        "recall_at_k",
        F.when(F.col("n_rel") > 0, F.col("tp") / F.col("n_rel")).otherwise(F.lit(0.0))
    )
)

precision_at_k = df_user_metrics.agg(F.avg("precision_at_k").alias("p")).first()["p"]
recall_at_k = df_user_metrics.agg(F.avg("recall_at_k").alias("r")).first()["r"]

# 6) Coverage@K = proportion d'items du catalogue recommandes au moins une fois
n_items_recommended = df_reco_eval_flat.select("movieId").distinct().count()
n_items_catalog = df_train_val.select("movieId").distinct().count()
coverage_at_k = n_items_recommended / n_items_catalog if n_items_catalog > 0 else 0.0

# 7) Tableau de synthese (ligne ALS baseline)
df_metrics_als = pd.DataFrame([
    {
        "method": "ALS",
        "k": K,
        "relevance_threshold": RELEVANCE_THRESHOLD,
        "rmse_test": float(rmse_test),
        "precision_at_k": float(precision_at_k),
        "recall_at_k": float(recall_at_k),
        "coverage_at_k": float(coverage_at_k),
        "n_eval_users": df_eval_users.count()
    }
])

display(df_metrics_als)

,method,k,relevance_threshold,rmse_test,precision_at_k,recall_at_k,coverage_at_k,n_eval_users
0,ALS,10,4.0,0.902036,0.011111,0.001381,0.01782,27


In [ ]:
spark.stop()
print("Session Spark arretee.")

Session Spark arretee.
